In [0]:
# Gold layer: dim_customers (Kimball customer dimension, Type 1 current-state)
# Source: b2b_ecommerce.silver.customers - silver is latest-state-only
#         (SCD Type 1 upsert, one row per customer_id, latest wins - no
#         history). SCD2 is intentionally NOT applied here: requirements.md
#         mandates version history only for dim_company (SCD Type 2
#         constraint, line 109), and none of the 15 KPIs needs
#         customer-attribute history - customer KPIs (CLV, orders per
#         customer, AOV, repeat rate) derive from order events, not from
#         changing customer attributes.
# Mechanism: AUTO CDC FROM SNAPSHOT into a Delta-backed streaming table
#         - published as a REAL table exactly like dim_companies
#         (user-directed 2026-11-18; the earlier materialized-view build
#         was dropped and rebuilt). Silver cannot be read as a change
#         stream (an upsert target has no change data feed), but it IS a
#         complete current-state snapshot - exactly what snapshot CDC
#         wants. The view is read at pipeline-run time and the run's
#         ingestion time is the snapshot version (ingestion-time pattern,
#         collect-free - SDP forbids driver-side collect()).
# MERGE / idempotency: keyed on customer_id (requirements' MERGE
#         constraint - safe to retry without duplicates). An identical
#         snapshot diffs to zero events, so retries and no-op runs change
#         nothing. A customer that disappears from silver is deleted here
#         too (snapshot diff semantics).
# Surrogate key: customer_sk = crc32(customer_id) - deterministic,
#         always non-negative and SHORT (unsigned 32-bit, max 10 digits;
#         user-directed 2026-11-18: short, easy key values, no 19-digit
#         hashes). customer_id is immutable, so the hash is stable
#         across refreshes; assigned at insert and never rewritten
#         (unlike dim_companies, no updated_at is mixed into the hash -
#         customers are Type 1, there are no versions to distinguish).
# Columns: only what the KPIs and the star schema need - company_id
#         (buyer-company affiliation: revenue/CLV by company, resolves to
#         dim_companies for as-of company attributes), email (matches
#         web-log auth_user for device conversion and is the only viable
#         lead-attribution key), gender + birth_date (demographics required
#         by requirements.md Source 1), is_active (standard dim attribute
#         for excluding inactive customers from activity KPIs).
# Excluded: first_name, last_name, phone, job_title, document_number - no
#         KPI uses them (phone was tested for lead attribution and adds
#         zero matches); created_at / updated_at - gold timestamp policy
#         (CDC plumbing and the mixed ART/UTC source clocks never reach
#         gold).
# Quality: no expectations - silver owns row-level validation, and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT flows
#         anyway (same decision as dim_companies).
# Layout: liquid-clustered by customer_sk (dims cluster by their surrogate
#         key - KPI views and facts join on it).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_CUSTOMERS = "b2b_ecommerce.silver.customers"


# Silver's complete current-state snapshot, plus the surrogate key.
# A batch temporary view: the engine reads it at each pipeline run
# (ingestion-time versioning). No driver-side actions anywhere.
@dp.temporary_view(name="customers_snapshot")
def customers_snapshot():
    return (
        spark.read.table(SILVER_CUSTOMERS)
        # select doubles as the exclusion list: names, phone, job_title,
        # document_number, created_at, updated_at stay out of gold
        .select(
            # deterministic surrogate key - stable across refreshes,
            # unique per immutable customer_id, non-negative and short
            # (crc32 is unsigned 32-bit: 0..4,294,967,295)
            F.crc32(F.col("customer_id").cast("string")).alias("customer_sk"),
            "customer_id",
            "company_id",
            "email",
            "gender",
            "birth_date",
            "is_active",
        )
    )


# Delta-backed streaming table: the AUTO CDC target, clustered on the
# surrogate key (dims cluster by their surrogate key). Published as a real
# table in b2b_ecommerce.gold, not a view.
dp.create_streaming_table(
    name="dim_customers",
    comment=(
        "Kimball customer dimension, SCD Type 1 (current state): one row per "
        "customer_id, MERGE-loaded from the b2b_ecommerce.silver.customers "
        "current-state snapshot via AUTO CDC FROM SNAPSHOT keyed on customer_id "
        "- identical snapshots diff to zero events, so retries are idempotent; "
        "rows removed from silver are deleted here. Deterministic surrogate key "
        "customer_sk = crc32(customer_id) (unsigned 32-bit, non-negative, "
        "short) is the join key carried by gold "
        "facts. Carries only KPI/star-schema columns: company_id (buyer-company "
        "affiliation), email (web-log auth_user match + lead attribution), "
        "gender/birth_date (required demographics), is_active. SCD2 deliberately "
        "not applied (requirements mandate it only for dim_company); "
        "created_at/updated_at and unused descriptive columns excluded per the "
        "gold timestamp policy."
    ),
    cluster_by=["customer_sk"],
)


# The auto-merge load: diff each snapshot against the last committed one
# and apply inserts/updates/deletes in place (Type 1 - no history columns).
dp.create_auto_cdc_from_snapshot_flow(
    target="dim_customers",
    source="customers_snapshot",  # ingestion-time snapshot versioning
    keys=["customer_id"],
    stored_as_scd_type=1,
)